# Cassava AI Root Cause Detective — Round 1 (LoRA fine-tune on Qwen2.5-1.5B-Instruct)

Run this notebook top-to-bottom on **Google Colab with a T4 GPU** (`Runtime > Change runtime type > T4 GPU`).

Round 1 goal: a real, rule-compliant, model-generated submission that beats the leaderboard benchmark
(needed to unlock Cassava's H200 notebooks for round 2). Everything here — the model actually generating
`\boxed{N}` answers via `model.generate()` — is compliant with the organizer's clarification that final
answers must come from running inference through the chosen ≤4B model, not a rule-based/threshold predictor.

**Before running:** upload `train.csv`, `validation_questions.csv`, `validation_target.csv`, `test.csv`,
`SampleSubmission.csv` when prompted in the data-loading cell below (or place them in `/content/` yourself).

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets huggingface_hub tqdm

## 2. Imports and global seed\n\nThe rules require reproducibility ("rerunning your model should always place you at the same position"), so every source of randomness is seeded here.

In [ ]:
import os
import random
import re
import math

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_DIR = "/content"
ADAPTER_DIR = "/content/qwen25-1.5b-aircd-lora"

## 2b. Hugging Face Hub auth (resumability)

Colab sessions can die mid-run. Section 8 below pushes the trained LoRA adapter to the Hub as soon as
training finishes, and skips training entirely (pulling the adapter back down instead) if it's already
there — so a dead session during the long inference loop costs you a Colab reconnect + re-running cells,
not a re-train. Add an `HF_TOKEN` secret in Colab (key icon in the left sidebar) with **write** access;
otherwise this cell falls back to an interactive login prompt.

In [ ]:
from huggingface_hub import login, HfApi, repo_exists

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

login(token=hf_token) if hf_token else login()

HF_USERNAME = HfApi().whoami()["name"]
HF_REPO_ID = f"{HF_USERNAME}/qwen25-1.5b-aircd-lora"
print(f"Adapter will be saved to https://huggingface.co/{HF_REPO_ID}")

## 3. Load data\n\nUpload the 5 competition CSVs if they aren't already in `/content/`.

In [ ]:
required = ["train.csv", "validation_questions.csv", "validation_target.csv", "test.csv", "SampleSubmission.csv"]
missing = [f for f in required if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    # Repo is public, so clone it straight into Colab first (no manual upload needed).
    clone_dir = "/content/cassava-network-anomaly-ai"
    if not os.path.exists(clone_dir):
        os.system(f"git clone --depth 1 'https://github.com/Ashuza11/cassava-network-anomaly-ai.git' {clone_dir}")
    for f in required:
        src = os.path.join(clone_dir, "data", f)
        if os.path.exists(src) and not os.path.exists(os.path.join(DATA_DIR, f)):
            os.system(f"cp {src} {DATA_DIR}/")
    missing = [f for f in required if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    try:
        from google.colab import files
        print(f"Clone didn't provide these, please upload: {missing}")
        uploaded = files.upload()
    except ImportError:
        raise RuntimeError(f"Missing data files and not running on Colab: {missing}")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_q_df = pd.read_csv(os.path.join(DATA_DIR, "validation_questions.csv"))
val_t_df = pd.read_csv(os.path.join(DATA_DIR, "validation_target.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "SampleSubmission.csv"))
print(train_df.shape, val_q_df.shape, val_t_df.shape, test_df.shape, sample_sub_df.shape)

## 4. Data prep utilities

Parses the embedded drive-test/engineering-parameter tables, converts train.csv's fixed-order `C1..C8`
questions into test.csv-style questions (random subset of options, random label prefix — calibrated to
match test.csv's actual distribution: ~79% of test rows keep the full 8 options with the rest at 5-7, and
about half use plain digit labels vs. a random LETTER+digit prefix), and synthesizes a short chain-of-thought
grounded in features computed directly from each question's own data tables (RB average, speed, handover
count, PCI-mod-30 collisions, RSRP comparisons, downtilt/beamwidth geometry). This is legitimate SFT-data
synthesis (distillation) — the organizer's rule against rule-based prediction applies to how *test* answers
are generated at inference time (via `model.generate()` below), not to how training targets are built.

In [ ]:
"""Convert train.csv's fixed-order C1..C8 questions into test.csv-style
shuffled, plain-numbered questions, with a synthetic grounded chain-of-thought
target ending in \\boxed{N}.

Pure pandas/regex — no GPU or model needed. Run directly (`python data_prep.py`)
to sanity-check parsing against the real CSVs.
"""
import math
import random
import re

import pandas as pd

C_OPTION_RE = re.compile(r"^C(\d+):\s*(.+)$", re.MULTILINE)
# test.csv/validation_questions.csv use a randomized label per question: plain
# digits ("1:"), or a single letter + digit ("M1:"), possibly with leading
# whitespace. Label is any alnum token so this covers all observed styles.
PLAIN_OPTION_RE = re.compile(r"^\s*([A-Za-z0-9]+):\s*(.+)$", re.MULTILINE)
OPTION_COUNT_RE = re.compile(r"following (\d+) potential (root causes|solutions)")

DRIVE_TABLE_MARKER = "User plane drive test data"
ENG_TABLE_MARKER = "Engeneering parameters data"

# Calibrated against test.csv's actual telecom-canonical subset: most questions
# show all 8 options, but a meaningful fraction show a random subset of 5-7.
SUBSET_SIZE_WEIGHTS = {8: 0.786, 6: 0.082, 5: 0.072, 7: 0.060}
# ~48% of test questions label options with plain digits; the rest use a
# single random uppercase letter + digit (e.g. "M1", "M2", ...).
PREFIX_DIGIT_PROB = 0.48
LETTERS = [chr(ord("A") + i) for i in range(26)]

NEIGHBOR_PCI_COLS = [
    f"Measurement PCell Neighbor Cell Top Set(Cell Level) Top {i} PCI" for i in range(1, 6)
]
NEIGHBOR_BRSRP_COLS = [
    f"Measurement PCell Neighbor Cell Top Set(Cell Level) Top {i} Filtered Tx BRSRP [dBm]"
    for i in range(1, 6)
]

CANONICAL_DESCRIPTIONS = {
    "C1": "The serving cell's downtilt angle is too large, causing weak coverage at the far end.",
    "C2": "The serving cell's coverage distance exceeds 1km, resulting in over-shooting.",
    "C3": "A neighboring cell provides higher throughput.",
    "C4": "Non-colocated co-frequency neighboring cells cause severe overlapping coverage.",
    "C5": "Frequent handovers degrade performance.",
    "C6": "Neighbor cell and serving cell have the same PCI mod 30, leading to interference.",
    "C7": "Test vehicle speed exceeds 40km/h, impacting user throughput.",
    "C8": "Average scheduled RBs are below 160, affecting throughput.",
}


# --------------------------------------------------------------------------
# Parsing
# --------------------------------------------------------------------------

def parse_options(question_text: str):
    """Return list of (label, description) in on-page order.

    label is 'C1'..'C8' for canonical (train.csv) questions, or the plain
    number string ('1', '2', ...) for shuffled test/validation questions.
    """
    c_matches = C_OPTION_RE.findall(question_text)
    if c_matches:
        return [(f"C{n}", desc.strip()) for n, desc in c_matches]

    header_end = len(question_text)
    for marker in ("\nGiven:", "\n" + DRIVE_TABLE_MARKER, "\n" + ENG_TABLE_MARKER):
        idx = question_text.find(marker)
        if idx != -1:
            header_end = min(header_end, idx)
    header = question_text[:header_end]
    return [(n, desc.strip()) for n, desc in PLAIN_OPTION_RE.findall(header)]


def _extract_table(question_text: str, marker: str):
    idx = question_text.find(marker)
    if idx == -1:
        return None
    lines = question_text[idx:].split("\n")[1:]
    table_lines, started = [], False
    for line in lines:
        if "|" in line:
            table_lines.append(line)
            started = True
        elif started:
            break
    if not table_lines:
        return None
    header = [c.strip() for c in table_lines[0].split("|")]
    rows = [l.split("|") for l in table_lines[1:]]
    df = pd.DataFrame(rows, columns=header).replace("-", pd.NA)
    return df


def parse_drive_test_table(question_text: str):
    return _extract_table(question_text, DRIVE_TABLE_MARKER)


def parse_engineering_params(question_text: str):
    return _extract_table(question_text, ENG_TABLE_MARKER)


# --------------------------------------------------------------------------
# Feature computation (for grounding synthetic CoT, not for prediction)
# --------------------------------------------------------------------------

def _num(series_or_val, default=None):
    v = pd.to_numeric(series_or_val, errors="coerce")
    return v


def _vertical_beamwidth(scenario):
    if not isinstance(scenario, str):
        return None
    s = scenario.strip().upper()
    if s == "DEFAULT" or re.match(r"SCENARIO_[1-5]$", s):
        return 6.0
    if re.match(r"SCENARIO_(6|7|8|9|10|11)$", s):
        return 12.0
    if re.match(r"SCENARIO_\d+$", s):
        return 25.0
    return None


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def compute_features(drive_df, eng_df):
    feats = {}
    if drive_df is None or drive_df.empty:
        return feats

    speed = _num(drive_df.get("GPS Speed (km/h)"))
    feats["avg_speed"] = round(speed.mean(), 1) if speed is not None and speed.notna().any() else None
    feats["max_speed"] = round(speed.max(), 1) if speed is not None and speed.notna().any() else None

    rb = _num(drive_df.get("5G KPI PCell Layer1 DL RB Num (Including 0)"))
    feats["avg_rb"] = round(rb.mean(), 1) if rb is not None and rb.notna().any() else None

    serving_pci = _num(drive_df.get("5G KPI PCell RF Serving PCI"))
    if serving_pci is not None and serving_pci.notna().any():
        transitions = int((serving_pci != serving_pci.shift()).sum()) - 1
        feats["handover_count"] = max(transitions, 0)
        mode = serving_pci.mode()
        dom_pci = int(mode.iloc[0]) if not mode.empty else None
    else:
        feats["handover_count"] = None
        dom_pci = None
    feats["serving_pci"] = dom_pci

    serving_rsrp = _num(drive_df.get("5G KPI PCell RF Serving SS-RSRP [dBm]"))
    feats["avg_serving_rsrp"] = (
        round(serving_rsrp.mean(), 1) if serving_rsrp is not None and serving_rsrp.notna().any() else None
    )

    neighbor_pairs = []
    for pcicol, brcol in zip(NEIGHBOR_PCI_COLS, NEIGHBOR_BRSRP_COLS):
        if pcicol in drive_df.columns and brcol in drive_df.columns:
            pcis = _num(drive_df[pcicol])
            brs = _num(drive_df[brcol])
            for pci, br in zip(pcis, brs):
                if pd.notna(pci) and pd.notna(br):
                    neighbor_pairs.append((int(pci), float(br)))

    feats["best_neighbor_brsrp"] = round(max(b for _, b in neighbor_pairs), 1) if neighbor_pairs else None

    collisions = set()
    if dom_pci is not None:
        dom_mod = dom_pci % 30
        collisions = {p for p, _ in neighbor_pairs if p != dom_pci and p % 30 == dom_mod}
    feats["pci_mod30_collisions"] = sorted(collisions)

    comparable = set()
    if feats["avg_serving_rsrp"] is not None:
        comparable = {p for p, b in neighbor_pairs if abs(b - feats["avg_serving_rsrp"]) <= 6}
    feats["comparable_strength_neighbors"] = sorted(comparable)

    feats["downtilt_total"] = None
    feats["vertical_beamwidth"] = None
    feats["height"] = None
    feats["distance_km"] = None
    feats["serving_gnodeb"] = None
    feats["non_colocated_overlap_pcis"] = []

    if eng_df is not None and not eng_df.empty and dom_pci is not None and "PCI" in eng_df.columns:
        eng_pci = _num(eng_df["PCI"])
        match = eng_df[eng_pci == dom_pci]
        if not match.empty:
            row = match.iloc[0]
            mech = _num(pd.Series([row.get("Mechanical Downtilt")])).iloc[0]
            dig = _num(pd.Series([row.get("Digital Tilt")])).iloc[0]
            dig_interp = None
            if pd.notna(dig):
                dig_interp = 6.0 if float(dig) == 255 else float(dig)
            if pd.notna(mech) and dig_interp is not None:
                feats["downtilt_total"] = round(float(mech) + dig_interp, 1)
            feats["vertical_beamwidth"] = _vertical_beamwidth(row.get("Beam Scenario"))
            height = _num(pd.Series([row.get("Height")])).iloc[0]
            feats["height"] = float(height) if pd.notna(height) else None
            feats["serving_gnodeb"] = row.get("gNodeB ID")

            lat_e = _num(pd.Series([row.get("Latitude")])).iloc[0]
            lon_e = _num(pd.Series([row.get("Longitude")])).iloc[0]
            lat_d = _num(drive_df.get("Latitude")).median() if drive_df.get("Latitude") is not None else None
            lon_d = _num(drive_df.get("Longitude")).median() if drive_df.get("Longitude") is not None else None
            if pd.notna(lat_e) and pd.notna(lon_e) and lat_d is not None and pd.notna(lat_d) and lon_d is not None and pd.notna(lon_d):
                feats["distance_km"] = round(haversine_km(float(lat_e), float(lon_e), float(lat_d), float(lon_d)), 3)

        if feats["serving_gnodeb"] is not None:
            eng_indexed = eng_df.assign(_pci=eng_pci)
            noncoloc = []
            for p in feats["comparable_strength_neighbors"]:
                rows = eng_indexed[eng_indexed["_pci"] == p]
                if not rows.empty and rows.iloc[0]["gNodeB ID"] != feats["serving_gnodeb"]:
                    noncoloc.append(p)
            feats["non_colocated_overlap_pcis"] = noncoloc

    return feats


# --------------------------------------------------------------------------
# Synthetic chain-of-thought
# --------------------------------------------------------------------------

def _fmt(v, suffix=""):
    return f"{v}{suffix}" if v is not None else "an unresolved value"


def synthesize_reasoning(true_label: str, feats: dict) -> str:
    f = feats or {}
    if true_label == "C1":
        return (
            f"The serving cell's total downtilt is about {_fmt(f.get('downtilt_total'), '°')} against a vertical "
            f"beamwidth of {_fmt(f.get('vertical_beamwidth'), '°')} at a mounting height of {_fmt(f.get('height'), 'm')}, "
            f"which narrows the effective coverage radius and pulls the main beam toward the ground close to the site. "
            f"The serving SS-RSRP averages {_fmt(f.get('avg_serving_rsrp'), ' dBm')}, consistent with weak far-end "
            f"coverage caused by excessive downtilt."
        )
    if true_label == "C2":
        return (
            f"The serving cell's total downtilt of about {_fmt(f.get('downtilt_total'), '°')} is shallow relative to "
            f"its vertical beamwidth of {_fmt(f.get('vertical_beamwidth'), '°')}, extending the main beam's reach well "
            f"beyond the intended ~1km coverage radius, indicating the cell is over-shooting its intended coverage "
            f"area rather than being properly contained."
        )
    if true_label == "C3":
        return (
            f"A neighboring cell reaches this location with a filtered Tx BRSRP of about "
            f"{_fmt(f.get('best_neighbor_brsrp'), ' dBm')}, comparable to or stronger than the serving cell's average "
            f"SS-RSRP of {_fmt(f.get('avg_serving_rsrp'), ' dBm')}, suggesting a neighbor would deliver higher "
            f"throughput than the current serving cell."
        )
    if true_label == "C4":
        pcis = f.get("non_colocated_overlap_pcis") or f.get("comparable_strength_neighbors") or []
        return (
            f"Neighbor cell(s) {pcis if pcis else '[not isolated from the data]'} arrive at comparable signal strength "
            f"to the serving cell but originate from a different, non-colocated site, producing severe overlapping "
            f"coverage that degrades throughput."
        )
    if true_label == "C5":
        return (
            f"The serving PCI changes {f.get('handover_count', 'several')} times across this short drive-test window, "
            f"indicating frequent handovers that interrupt scheduling and degrade sustained throughput."
        )
    if true_label == "C6":
        collisions = f.get("pci_mod30_collisions") or []
        return (
            f"The serving cell (PCI {f.get('serving_pci', '?')}) shares PCI mod 30 with neighbor PCI(s) "
            f"{collisions if collisions else '[not isolated from the data]'}, causing reference-signal collision and "
            f"interference that suppresses throughput."
        )
    if true_label == "C7":
        return (
            f"The test vehicle's GPS speed averages {_fmt(f.get('avg_speed'), ' km/h')} and peaks at "
            f"{_fmt(f.get('max_speed'), ' km/h')}, with the peak exceeding the 40km/h mobility threshold at which "
            f"scheduling and channel estimation degrade, impacting throughput."
        )
    if true_label == "C8":
        return (
            f"The average scheduled RB count is about {_fmt(f.get('avg_rb'))}, below the 160-RB threshold needed to "
            f"sustain high throughput, directly explaining the low observed data rate."
        )
    return "Based on the provided data, this is the most consistent root cause."


# --------------------------------------------------------------------------
# Reformatting train.csv rows into test-style shuffled questions
# --------------------------------------------------------------------------

def _choose_subset_size(rng: random.Random) -> int:
    sizes = list(SUBSET_SIZE_WEIGHTS.keys())
    weights = list(SUBSET_SIZE_WEIGHTS.values())
    return rng.choices(sizes, weights=weights, k=1)[0]


def _choose_prefix_style(rng: random.Random):
    """Returns None for plain-digit labels, else a single uppercase letter
    used as a "<letter><digit>" prefix (e.g. 'M' -> 'M1', 'M2', ...)."""
    if rng.random() < PREFIX_DIGIT_PROB:
        return None
    return rng.choice(LETTERS)


def reformat_example(question_text: str, true_label: str, rng: random.Random):
    """Mimic test.csv's real distribution: a random subset of the 8 canonical
    options (always including the true label) in random order, labeled with
    a randomly chosen prefix style (plain digits or LETTER+digit)."""
    matches = list(C_OPTION_RE.finditer(question_text))
    if not matches:
        raise ValueError("no canonical C-prefixed options found in question")

    labels = [f"C{m.group(1)}" for m in matches]
    descs = [m.group(2).strip() for m in matches]
    label_to_desc = dict(zip(labels, descs))

    subset_size = _choose_subset_size(rng)
    distractors = [lbl for lbl in labels if lbl != true_label]
    chosen_distractors = rng.sample(distractors, min(subset_size - 1, len(distractors)))
    subset_labels = chosen_distractors + [true_label]
    rng.shuffle(subset_labels)

    prefix = _choose_prefix_style(rng)
    new_position = {}
    new_lines = []
    for i, lbl in enumerate(subset_labels, start=1):
        pos_str = f"{prefix}{i}" if prefix else str(i)
        new_position[lbl] = pos_str
        new_lines.append(f"{pos_str}: {label_to_desc[lbl]}")
    new_block = "\n".join(new_lines)

    start, end = matches[0].start(), matches[-1].end()
    new_question = question_text[:start] + new_block + question_text[end:]
    new_question = OPTION_COUNT_RE.sub(
        lambda m: f"following {len(subset_labels)} potential {m.group(2)}", new_question, count=1
    )

    true_position = new_position[true_label]

    try:
        drive_df = parse_drive_test_table(question_text)
        eng_df = parse_engineering_params(question_text)
        feats = compute_features(drive_df, eng_df)
    except Exception:
        feats = {}

    reasoning = synthesize_reasoning(true_label, feats)
    target_completion = f"{reasoning}\n\\boxed{{{true_position}}}"
    return new_question, target_completion


def build_sft_dataset(train_csv_path: str, seed: int = 42) -> pd.DataFrame:
    df = pd.read_csv(train_csv_path)
    rng = random.Random(seed)
    rows = []
    for _, r in df.iterrows():
        true_label = str(r["answer"]).strip()
        try:
            new_q, target = reformat_example(r["question"], true_label, rng)
        except Exception as e:
            print(f"skip {r['ID']}: {e}")
            continue
        rows.append({"ID": r["ID"], "prompt": new_q, "completion": target})
    return pd.DataFrame(rows)

## 5. Scoring utilities (boxed-answer extraction, position→label mapping, Pass@1)

In [ ]:
"""Answer extraction and Pass@1 scoring against validation_target.csv.

Pure regex/pandas — no GPU or model needed.
"""
import re

import pandas as pd

BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")


def extract_boxed(text: str):
    if not isinstance(text, str):
        return None
    matches = BOXED_RE.findall(text)
    if not matches:
        return None
    return matches[-1].strip()


def map_position_to_label(question_text: str, canonical_descriptions=None):
    """{position_str: canonical_label} for a shuffled test/validation question,
    by matching each option's description text against the known canonical wording.
    Falls back to identity mapping (position -> position) for questions whose
    options don't match any canonical description (e.g. generic knowledge MCQs,
    where the 'label' is just its own position number).
    """
    canonical_descriptions = canonical_descriptions or CANONICAL_DESCRIPTIONS
    desc_to_label = {desc.strip(): label for label, desc in canonical_descriptions.items()}
    options = parse_options(question_text)
    mapping = {}
    for pos, desc in options:
        mapping[pos] = desc_to_label.get(desc.strip(), pos)
    return mapping


def pass_at_1(pred_df: pd.DataFrame, target_df: pd.DataFrame, question_lookup: dict,
              canonical_descriptions=None) -> float:
    """pred_df: columns ID (suffixed _1.._4), Target (generated text containing \\boxed{N})
    target_df: columns ID (suffixed _1.._4), Target (ground-truth canonical label, e.g. 'C2')
    question_lookup: {base_id (no suffix): question_text}
    """
    merged = pred_df.merge(target_df, on="ID", suffixes=("_pred", "_true"))
    correct, total = 0, 0
    for _, row in merged.iterrows():
        base_id = row["ID"].rsplit("_", 1)[0]
        qtext = question_lookup.get(base_id)
        if qtext is None:
            continue
        pos_map = map_position_to_label(qtext, canonical_descriptions)
        boxed = extract_boxed(row["Target_pred"])
        pred_label = pos_map.get(boxed)
        total += 1
        if pred_label == row["Target_true"]:
            correct += 1
    return correct / total if total else 0.0

## 6. Build the SFT dataset from train.csv

In [ ]:
sft_df = build_sft_dataset(os.path.join(DATA_DIR, "train.csv"), seed=SEED)
print(f"Built {len(sft_df)} / {len(train_df)} SFT examples")
print(sft_df.iloc[0]["prompt"][:400])
print("---")
print(sft_df.iloc[0]["completion"])

In [ ]:
def to_conversational(row):
    return {
        "prompt": [{"role": "user", "content": row["prompt"]}],
        "completion": [{"role": "assistant", "content": row["completion"]}],
    }

sft_records = [to_conversational(r) for _, r in sft_df.iterrows()]
train_dataset = Dataset.from_list(sft_records)
train_dataset[0]

## 7. Load base model in 4-bit + attach LoRA

Using `bnb_4bit_compute_dtype=torch.bfloat16` to match Qwen2.5's native checkpoint dtype (`bfloat16`),
and training with `bf16=True` below instead of `fp16=True`. bf16 needs no loss scaling, so
`torch.cuda.amp.GradScaler` is never invoked — this avoids `NotImplementedError` crashes from any
bf16-dtype adapter/model parameters hitting the fp16-only scaler. Colab's free-tier T4 (Turing) lacks
bf16 tensor-core acceleration, so this runs somewhat slower than true fp16 would on that GPU, but it
runs correctly — no A100/H100 required.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,  # matches bnb_4bit_compute_dtype and bf16=True training below
    device_map="auto",
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

## 8. Train (LoRA SFT, completion-only loss)

In [ ]:
if repo_exists(HF_REPO_ID):
    print(f"Adapter already on the Hub ({HF_REPO_ID}) — loading it instead of retraining.")
    model = PeftModel.from_pretrained(model, HF_REPO_ID)
else:
    sft_config = SFTConfig(
        output_dir=ADAPTER_DIR,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        max_length=2048,
        logging_steps=20,
        save_strategy="epoch",
        bf16=True,
        fp16=False,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        peft_config=peft_config,
    )
    trainer.train()
    model = trainer.model  # the actual PeftModel wrapper (adapter + frozen base)
    trainer.save_model(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)

    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f"Pushed adapter to https://huggingface.co/{HF_REPO_ID}")

## 8b. Reload merged bf16 model for fast inference

Regardless of whether the cell above just trained or pulled the adapter from the Hub, reload the base
model unquantized in bf16 and merge the LoRA adapter into it. This sidesteps bitsandbytes' slower
per-token 4-bit dequantization during generation (a 1.5B model in bf16 is only ~3GB, well within a T4's
16GB), and reloading straight from `HF_REPO_ID` means a fresh Colab session that died mid-inference can
skip retraining entirely — just rerun cells 1-8 (fast) then this cell, and resume generation.

In [ ]:
del model
torch.cuda.empty_cache()

inference_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map="auto",
)
model = PeftModel.from_pretrained(inference_base, HF_REPO_ID)
model = model.merge_and_unload()
model.eval()

## 9. Generation helper

4 independent generations per question with fixed, index-derived seeds (reproducible, as the rules require).
Low-moderate temperature since Pass@1 here is *averaged over the 4 samples independently* (not "best of 4"),
so each individual generation should be as accurate as possible rather than deliberately diversified.

In [ ]:
model.eval()
GEN_TEMPERATURE = 0.4
GEN_TOP_P = 0.9
GEN_MAX_NEW_TOKENS = 300

def generate_answers(question_text, num_samples=4, base_seed=SEED):
    messages = [{"role": "user", "content": question_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    set_seed(base_seed)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=num_samples,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in out]

## 10. Local validation (the real checkpoint gate)

Don't spend a Zindi submission until this number looks meaningfully better than guessing
(1/8 ≈ 12.5% for the common 8-option case). This only covers the "full canonical 8-option, plain-digit"
slice of the distribution, since that's all `validation_questions.csv` contains — the real leaderboard
score will also include test.csv's harder slices (random subsets/letter-prefixes of the same 8 categories,
a disjoint LTE-style 9-category fault domain never seen in training, and generic-knowledge MCQs), so treat
this as an optimistic upper bound, not the expected leaderboard score.

In [ ]:
from tqdm.auto import tqdm

def run_predictions(question_df, num_samples=4, base_seed=SEED):
    rows = []
    for _, r in tqdm(question_df.iterrows(), total=len(question_df), desc="Generating"):
        gens = generate_answers(r["question"], num_samples=num_samples, base_seed=base_seed + hash(r["ID"]) % 10_000)
        for i, g in enumerate(gens, start=1):
            rows.append({"ID": f"{r['ID']}_{i}", "Target": g})
    return pd.DataFrame(rows)

val_pred_df = run_predictions(val_q_df)
val_lookup = dict(zip(val_q_df["ID"], val_q_df["question"]))
score = pass_at_1(val_pred_df, val_t_df, val_lookup)
print(f"Local validation Pass@1: {score:.4f}")

## 11. Generate test.csv predictions and write submission.csv

In [ ]:
test_pred_df = run_predictions(test_df)
assert set(test_pred_df["ID"]) == set(sample_sub_df["ID"]), "ID mismatch vs SampleSubmission.csv"
test_pred_df = test_pred_df.set_index("ID").loc[sample_sub_df["ID"]].reset_index()
test_pred_df.to_csv("/content/submission.csv", index=False)
test_pred_df.head()

## 12. Round 2+ (not built here)

- Mix in generic MCQ/math examples to shore up knowledge retention.
- Synthesize training-style examples for the disjoint LTE-style 9-category fault domain seen in ~12% of
  test.csv (columns like CCE Fail Rate / Avg MCS / BLER%, categories A-I) — currently zero training coverage.
- Replace the templated CoT with physics-grounded reasoning for the geometry-heavy categories (C1/C2/C4)
  once verified against train.csv's known labels.
- Sweep LoRA rank/epochs, try alternate ≤4B base models, or move to full fine-tuning once H200 access is
  unlocked.